requirements.txt


In [ ]:
streamlit>=1.20.0
pandas>=1.5.0
pytest>=7.0.0

core/__init__.py

core/parser.py

In [ ]:
import ast

class CodeParser:
    """Parses target Python script files or raw code strings into an Abstract Syntax Tree (AST)."""

    @staticmethod
    def parse_file(file_path: str) -> ast.AST:
        with open(file_path, "r", encoding="utf-8") as file:
            source_code = file.read()
        return ast.parse(source_code)

    @staticmethod
    def parse_string(source_code: str) -> ast.AST:
        return ast.parse(source_code)

core/rules.py

In [ ]:
import ast

class LookaheadVisitor(ast.NodeVisitor):
    """AST Visitor that scans syntax trees for lookahead bias and negative Pandas shifts."""
    def __init__(self):
        self.violations = []

    def visit_BinOp(self, node: ast.BinOp):
        if isinstance(node.op, ast.Add):
            if isinstance(node.right, ast.Constant) and isinstance(node.right.value, int) and node.right.value > 0:
                if isinstance(node.left, ast.Name):
                    self.violations.append({
                        "line": getattr(node, 'lineno', 'Unknown'),
                        "type": "Lookahead Indexing Bias",
                        "detail": f"Illegal future offset (+{node.right.value}) added to time index '{node.left.id}'."
                    })
        self.generic_visit(node)

    def visit_Call(self, node: ast.Call):
        if isinstance(node.func, ast.Attribute) and node.func.attr == "shift":
            for arg in node.args:
                if isinstance(arg, ast.UnaryOp) and isinstance(arg.op, ast.USub):
                    if isinstance(arg.operand, ast.Constant):
                        self.violations.append({
                            "line": getattr(node, 'lineno', 'Unknown'),
                            "type": "Pandas Data Leakage",
                            "detail": f"Negative shift shift(-{arg.operand.value}) references future prices in backtest."
                        })
        self.generic_visit(node)

core/auditor.py

In [ ]:
import ast
from core.parser import CodeParser
from core.rules import LookaheadVisitor

class QuantAuditor:
    """Orchestrates static analysis audits on quantitative strategy code."""

    def __init__(self):
        self.parser = CodeParser()

    def audit_file(self, file_path: str) -> dict:
        try:
            tree = self.parser.parse_file(file_path)
            return self._evaluate_ast(tree)
        except SyntaxError as e:
            return {"status": "SYNTAX_ERROR", "errors": [{"line": e.lineno, "type": "SyntaxError", "detail": str(e)}]}

    def audit_string(self, source_code: str) -> dict:
        try:
            tree = self.parser.parse_string(source_code)
            return self._evaluate_ast(tree)
        except SyntaxError as e:
            return {"status": "SYNTAX_ERROR", "errors": [{"line": e.lineno, "type": "SyntaxError", "detail": str(e)}]}

    def _evaluate_ast(self, tree: ast.AST) -> dict:
        visitor = LookaheadVisitor()
        visitor.visit(tree)
        if visitor.violations:
            return {"status": "FAILED", "errors": visitor.violations}
        return {"status": "PASSED", "errors": []}

main.py



In [ ]:
import sys
import argparse
from core.auditor import QuantAuditor

def main():
    parser = argparse.ArgumentParser(description="LeakageGuard AI - Quantitative Code Auditor")
    parser.add_argument("--file", type=str, help="Path to the Python trading strategy script")
    args = parser.parse_args()

    if not args.file:
        print("Error: Please specify a file using --file <path>")
        sys.exit(1)

    auditor = QuantAuditor()
    print(f"[*] Auditing file: {args.file} ...\n")
    result = auditor.audit_file(args.file)

    if result["status"] == "PASSED":
        print("✅ PASSED: No lookahead bias or data leakage detected!")
    elif result["status"] == "FAILED":
        print("❌ FAILED: Security violations found:")
        for err in result["errors"]:
            print(f"  - [Line {err['line']}] [{err['type']}]: {err['detail']}")

if __name__ == "__main__":
    main()

app.py

In [ ]:
import streamlit as st
from core.auditor import QuantAuditor

st.set_page_config(page_title="LeakageGuard AI", page_icon="🛡️", layout="wide")
st.title("🛡️ LeakageGuard AI: Quant Code Auditor")
st.markdown("Automated Static Analysis to Catch **Lookahead Bias** and **Data Leakage** in Trading Strategies.")

auditor = QuantAuditor()

default_code = """import pandas as pd

def trading_strategy(df):
    # Code with intentional lookahead bias for testing
    df['future_price'] = df['close'].shift(-1)

    for t in range(len(df)-1):
        if df['close'].iloc[t + 1] > df['close'].iloc[t]:
            print("BUY")
"""

source_code = st.text_area("Paste Python Strategy Code Here:", value=default_code, height=250)

if st.button("Audit Strategy Code"):
    result = auditor.audit_string(source_code)
    if result["status"] == "PASSED":
        st.success("✅ PASSED: No lookahead bias or time-series data leakage detected!")
    elif result["status"] == "FAILED":
        st.error("❌ AUDIT FAILED: Lookahead Violations Detected!")
        for idx, err in enumerate(result["errors"], 1):
            st.warning(f"**Violation #{idx}** (Line {err['line']}) — **{err['type']}**: {err['detail']}")

examples/flawed_strategy.py

In [ ]:
import pandas as pd

def generate_signals(df):
    df['future_close'] = df['close'].shift(-1)
    if df['close'].iloc[t + 1] > df['close'].iloc[t]:
        return "BUY"

examples/clean_strategy.py

In [ ]:
import pandas as pd

def generate_signals(df):
    df['sma_20'] = df['close'].rolling(window=20).mean()
    return df['sma_20']

tests/test_auditor.py

In [ ]:
import unittest
from core.auditor import QuantAuditor

class TestQuantAuditor(unittest.TestCase):
    def test_flawed_code_fails(self):
        auditor = QuantAuditor()
        result = auditor.audit_string("future = close.shift(-1)\nval = close[t + 1]")
        self.assertEqual(result["status"], "FAILED")

    def test_clean_code_passes(self):
        auditor = QuantAuditor()
        result = auditor.audit_string("sma = close.rolling(20).mean()")
        self.assertEqual(result["status"], "PASSED")

if __name__ == "__main__":
    unittest.main()